# 12 - A2A Protocol
Implement agent-to-agent communication using A2A: define AgentCards, set up supervisor routing, and demonstrate multi-agent task delegation.

## Step 1: Load Configuration

In [ ]:
import boto3
import json
import asyncio
from pathlib import Path

config = json.loads(Path("config.json").read_text())
REGION = config["REGION"]
PREFIX = config["PROJECT_PREFIX"]
MODEL_ID = config["MODEL_ID"]
GRAPH_ID = config["NEPTUNE_GRAPH_ID"]
KB_ID = config["KB_ID"]

print(f"\u2713 Config loaded (region={REGION}, prefix={PREFIX})")

## Step 2: Define Agent Cards

AgentCards describe agent capabilities, enabling discovery and routing.

In [ ]:
from strands.multiagent.a2a import A2AServer
from strands import Agent, tool
from strands.models import BedrockModel

# Define AgentCards for each specialist agent
search_agent_card = {
    "name": f"{PREFIX}-search",
    "description": "Searches the knowledge graph for experts, topics, and documents",
    "capabilities": ["expert-search", "document-search", "topic-search"],
    "skills": [
        {"name": "find_experts", "description": "Find people with expertise in a given topic"},
        {"name": "search_docs", "description": "Search organizational documents"},
        {"name": "get_profile", "description": "Get a person's expertise profile"}
    ],
    "endpoint": f"http://localhost:8080/a2a",
    "protocol_version": "1.0"
}

scoring_agent_card = {
    "name": f"{PREFIX}-scoring",
    "description": "Computes expertise scores using graph algorithms (PageRank, centrality)",
    "capabilities": ["scoring", "ranking", "graph-analytics"],
    "skills": [
        {"name": "compute_pagerank", "description": "Run PageRank on the knowledge graph"},
        {"name": "compute_centrality", "description": "Calculate centrality scores"},
        {"name": "get_leaderboard", "description": "Get expertise leaderboard by topic"}
    ],
    "endpoint": f"http://localhost:8081/a2a",
    "protocol_version": "1.0"
}

advisor_agent_card = {
    "name": f"{PREFIX}-advisor",
    "description": "Provides recommendations for team composition and knowledge gaps",
    "capabilities": ["recommendations", "gap-analysis", "team-building"],
    "skills": [
        {"name": "recommend_team", "description": "Suggest optimal team for a project"},
        {"name": "identify_gaps", "description": "Find knowledge gaps in the org"},
        {"name": "suggest_training", "description": "Recommend training based on gaps"}
    ],
    "endpoint": f"http://localhost:8082/a2a",
    "protocol_version": "1.0"
}

agent_cards = [search_agent_card, scoring_agent_card, advisor_agent_card]
print(f"\u2713 {len(agent_cards)} AgentCards defined")
for card in agent_cards:
    print(f"  - {card['name']}: {card['description'][:50]}...")

## Step 3: Build Specialist Agents

Create the search agent as a concrete implementation.

In [ ]:
neptune = boto3.client("neptune-graph", region_name=REGION)
model = BedrockModel(model_id=MODEL_ID, region_name=REGION)

@tool
def find_experts(topic: str) -> str:
    """Find experts on a given topic from the knowledge graph.
    
    Args:
        topic: Topic to search for experts in
    """
    query = """
    MATCH (p:Person)-[r:KNOWS_ABOUT]->(t:Topic)
    WHERE toLower(t.name) CONTAINS toLower($topic)
    RETURN p.name AS name, p.title AS title, r.strength AS strength
    ORDER BY r.strength DESC LIMIT 5
    """
    response = neptune.execute_query(
        graphIdentifier=GRAPH_ID,
        queryString=query,
        language="OPEN_CYPHER",
        parameters={"topic": {"value": topic}}
    )
    results = json.loads(response["payload"].read())
    output = [f"{r['name']} ({r['title']}) - strength: {r['strength']}" for r in results["results"]]
    return "\n".join(output) if output else "No experts found"

@tool
def compute_pagerank() -> str:
    """Run PageRank algorithm on the knowledge graph to score expertise."""
    query = """
    MATCH (p:Person)-[r:KNOWS_ABOUT]->(t:Topic)
    WITH p, sum(r.strength) AS total_strength, count(t) AS topic_count
    SET p.pagerank = total_strength * topic_count / 10.0
    RETURN p.name AS name, p.pagerank AS score
    ORDER BY score DESC
    """
    response = neptune.execute_query(
        graphIdentifier=GRAPH_ID,
        queryString=query,
        language="OPEN_CYPHER"
    )
    results = json.loads(response["payload"].read())
    output = [f"{r['name']}: {r['score']:.3f}" for r in results["results"]]
    return "PageRank scores:\n" + "\n".join(output)

# Create specialist agents
search_agent = Agent(
    model=model,
    system_prompt="You are the OrgIQ Search agent. Find experts and documents in the knowledge graph.",
    tools=[find_experts]
)

scoring_agent = Agent(
    model=model,
    system_prompt="You are the OrgIQ Scoring agent. Compute expertise scores using graph algorithms.",
    tools=[compute_pagerank]
)

print("\u2713 Specialist agents created")
print("  - search_agent: find_experts")
print("  - scoring_agent: compute_pagerank")

## Step 4: Build Supervisor Agent with Routing

The supervisor reads AgentCards and routes tasks to specialists.

In [ ]:
# Supervisor routing tool
@tool
def delegate_to_agent(agent_name: str, task: str) -> str:
    """Delegate a task to a specialist agent based on their capabilities.
    
    Args:
        agent_name: Name of the agent to delegate to (orgiq-search or orgiq-scoring)
        task: The task description to send to the agent
    """
    # In production, this would use A2A protocol over HTTP
    # For notebook, we route directly to agent instances
    agents = {
        f"{PREFIX}-search": search_agent,
        f"{PREFIX}-scoring": scoring_agent,
    }
    
    target = agents.get(agent_name)
    if not target:
        return f"Unknown agent: {agent_name}. Available: {list(agents.keys())}"
    
    response = target(task)
    return f"[Response from {agent_name}]:\n{response}"

# Format agent cards as context for supervisor
cards_context = json.dumps(agent_cards, indent=2)

supervisor = Agent(
    model=model,
    system_prompt=f"""You are the OrgIQ Supervisor agent. You route user requests to specialist agents.

Available agents:
{cards_context}

When a user asks a question:
1. Analyze which agent(s) are best suited based on their capabilities
2. Delegate using delegate_to_agent with the agent name and task
3. Synthesize responses from multiple agents if needed
4. Present a unified answer to the user""",
    tools=[delegate_to_agent]
)

print("\u2713 Supervisor agent created with routing capability")

## Step 5: Test Single-Agent Delegation

In [ ]:
# Test: Supervisor delegates to search agent
response = supervisor("Who are our top experts in machine learning?")
print("Supervisor response (single delegation):")
print(response)

## Step 6: Test Multi-Agent Delegation

In [ ]:
# Test: Supervisor needs both search and scoring
response = supervisor(
    "I need a comprehensive expertise report: who are the top experts and what are their scores? "
    "First find the experts, then compute their rankings."
)
print("Supervisor response (multi-agent delegation):")
print(response)

## Step 7: A2A Server Setup

Show how to expose an agent as an A2A server for inter-agent communication.

In [ ]:
# Define A2A server for the search agent
# In production, this runs inside the AgentCore Runtime container

a2a_server = A2AServer(
    agent=search_agent,
    agent_card=search_agent_card,
    host="0.0.0.0",
    port=8080
)

print("\u2713 A2A Server configured")
print(f"  Agent: {search_agent_card['name']}")
print(f"  Endpoint: {search_agent_card['endpoint']}")
print(f"  Capabilities: {search_agent_card['capabilities']}")
print()
print("  To start server (in agent container):")
print("    a2a_server.start()")
print()
print("  A2A endpoints exposed:")
print("    GET  /a2a/.well-known/agent-card  - Agent discovery")
print("    POST /a2a/tasks                   - Create task")
print("    GET  /a2a/tasks/{id}              - Get task status")
print("    POST /a2a/tasks/{id}/cancel       - Cancel task")

## Step 8: A2A Task Lifecycle

In [ ]:
# Demonstrate A2A task lifecycle
import uuid

# A2A Task structure
a2a_task = {
    "id": str(uuid.uuid4()),
    "type": "task",
    "status": "pending",
    "from_agent": f"{PREFIX}-supervisor",
    "to_agent": f"{PREFIX}-search",
    "input": {
        "messages": [
            {"role": "user", "content": "Find all Kubernetes experts"}
        ]
    },
    "metadata": {
        "priority": "normal",
        "timeout_seconds": 30
    }
}

print("A2A Task Lifecycle:")
print(f"\n1. CREATED (by supervisor):")
print(f"   Task ID: {a2a_task['id'][:12]}...")
print(f"   From: {a2a_task['from_agent']}")
print(f"   To: {a2a_task['to_agent']}")

# Simulate execution
a2a_task["status"] = "in_progress"
print(f"\n2. IN_PROGRESS:")
print(f"   Status: {a2a_task['status']}")

# Execute the task
task_input = a2a_task["input"]["messages"][0]["content"]
result = search_agent(task_input)

a2a_task["status"] = "completed"
a2a_task["output"] = {"messages": [{"role": "assistant", "content": str(result)}]}
print(f"\n3. COMPLETED:")
print(f"   Status: {a2a_task['status']}")
print(f"   Result: {str(result)[:100]}...")

print("\n\u2713 A2A task lifecycle demonstrated")

## Step 9: Agent Discovery Pattern

In [ ]:
# Agent discovery - how supervisor finds specialist agents
def discover_agents(required_capabilities: list) -> list:
    """Find agents that match required capabilities."""
    matching = []
    for card in agent_cards:
        overlap = set(required_capabilities) & set(card["capabilities"])
        if overlap:
            matching.append({
                "name": card["name"],
                "matched_capabilities": list(overlap),
                "endpoint": card["endpoint"]
            })
    return matching

# Test discovery
print("Discovery: agents with 'expert-search' capability:")
results = discover_agents(["expert-search"])
for r in results:
    print(f"  {r['name']} (matched: {r['matched_capabilities']})")

print("\nDiscovery: agents with 'scoring' or 'ranking':")
results = discover_agents(["scoring", "ranking"])
for r in results:
    print(f"  {r['name']} (matched: {r['matched_capabilities']})")

print("\nDiscovery: agents with 'recommendations':")
results = discover_agents(["recommendations"])
for r in results:
    print(f"  {r['name']} (matched: {r['matched_capabilities']})")

print("\n\u2713 Agent discovery working")

## Step 10: Parallel Task Delegation

In [ ]:
# Demonstrate parallel task delegation
import concurrent.futures

def execute_task(agent, task):
    """Execute a task on an agent (simulating A2A call)."""
    return str(agent(task))

tasks = [
    (search_agent, "Find authentication experts"),
    (scoring_agent, "Compute PageRank scores for all people"),
]

print("\u231b Executing parallel A2A tasks...")
with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
    futures = {
        executor.submit(execute_task, agent, task): task
        for agent, task in tasks
    }
    
    for future in concurrent.futures.as_completed(futures):
        task_desc = futures[future]
        result = future.result()
        print(f"\n  Task: {task_desc}")
        print(f"  Result: {result[:100]}...")

print("\n\u2713 Parallel delegation complete")

## Summary

- [x] AgentCards defined for 3 specialist agents
- [x] Specialist agents built (search, scoring)
- [x] Supervisor agent with routing logic
- [x] Single-agent delegation tested
- [x] Multi-agent delegation tested
- [x] A2A Server setup demonstrated
- [x] A2A task lifecycle (pending -> in_progress -> completed)
- [x] Agent discovery by capabilities
- [x] Parallel task delegation

**Next:** [13-scoring.ipynb](13-scoring.ipynb) — Run graph algorithms for expertise scoring